# CFTC 14 日映射时点影响核对
只读已封存 Qlib Recorder，核对 2025-12-30 是否用到当前页面标记 12-31 更新的报告行。

In [1]:
import hashlib,json,os
from pathlib import Path
import pandas as pd,qlib
from qlib.workflow import R
root=Path.cwd().resolve();card=root/'research/stage1_cftc_tff_2025_backlog_update_audit_20261023';plan=json.loads((card/'map_impact_plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(card/'result.json')==plan['source_result_sha256']
mc=root/plan['map_summary'];summary=json.loads(mc.read_text());assert sha(mc)==plan['map_summary_sha256'] and summary['recorder_id']==plan['map_recorder_id'] and summary['experiment_id']==plan['map_experiment_id'] and summary['nas_readback']['verified']
artifact=Path(summary['recorder_uri'])/summary['experiment_id']/summary['recorder_id']/'artifacts'/plan['map_artifact'];assert sha(artifact)==plan['map_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(summary['recorder_uri']):mapped=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id']).load_object(plan['map_artifact'])
subset=mapped.loc[pd.to_datetime(mapped.signal_date).isin(pd.to_datetime(plan['exact_signal_dates']))].copy();assert len(subset)==4
rows=[{'signal_date':str(x.signal_date.date()),'source_report_date':str(x.report_date.date()),'source_available_at_utc':str(x.available_at_utc),'source_known':bool(x.source_known)} for x in subset.itertuples(index=False)]
assert [x['signal_date'] for x in rows]==plan['exact_signal_dates']
problem=next(x for x in rows if x['signal_date']=='2025-12-30')
assert problem['source_report_date']=='2025-12-23' and problem['source_known']
result={'status':'existing_CFTC_14d_map_uses_current_Dec23_row_on_Dec30_before_Dec31_update','plan_sha256':sha(card/'map_impact_plan.json'),'source_result_sha256':sha(card/'result.json'),'map_summary_sha256':sha(mc),'map_recorder_id':summary['recorder_id'],'mapped_rows':rows,'qualification':'needs_timing_repair_before_model_input','not_first_public_proof':True,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_writable_Recorders':0}
(card/'map_impact.json').write_text(json.dumps(result,indent=2)+'\n')
print(result['status'],rows)

2026/10/23 05:20:50 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[92128:MainThread](2026-10-23 05:20:51,032) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[92128:MainThread](2026-10-23 05:20:51,034) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[92128:MainThread](2026-10-23 05:20:51,034) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


existing_CFTC_14d_map_uses_current_Dec23_row_on_Dec30_before_Dec31_update [{'signal_date': '2025-12-29', 'source_report_date': '2025-12-16', 'source_available_at_utc': '2025-12-24 05:00:00+00:00', 'source_known': True}, {'signal_date': '2025-12-30', 'source_report_date': '2025-12-23', 'source_available_at_utc': '2025-12-30 05:00:00+00:00', 'source_known': True}, {'signal_date': '2025-12-31', 'source_report_date': '2025-12-23', 'source_available_at_utc': '2025-12-30 05:00:00+00:00', 'source_known': True}, {'signal_date': '2026-01-02', 'source_report_date': '2025-12-23', 'source_available_at_utc': '2025-12-30 05:00:00+00:00', 'source_known': True}]


## 中文结论
# CFTC 2025 积压报告：计划发布与当前周报更新时点

固定三份 CFTC 官方页面（12 月 9 日特殊公告、2025-12-02 和 2025-12-23 两期 TFF Futures Only 周报），各请求一次，原始正文/响应头、UTC 时刻和哈希已在本机与 Yanxuan NAS 留存，远端哈希及限定逐文件恢复通过。响应头含临时 Cookie，原字节不提交 Git。0 市场收益读取、0 拟合、0 账户。

12 月 9 日公告表把 12-02 报告安排在 12-17 发布，其当前周报页标记 `Updated December 17, 2025`。12-23 报告安排 12-29 发布，但当前页标记 `Updated December 31, 2025`。`Updated` 不是首次公开证明；它说明当前页面版本晚于计划 12-30 的 Stage1 可用时点，无法认定当前年度文件中的该行在 12-30 已同值公开。

对已封存的 Qlib 14 日映射做一次只读定向核验：2025-12-30 信号确实选用了 12-23 报告，`source_known=true`，可用时点写为 12-30 00:00 美东。因此现有映射**在进入任何信息拟合前必须修复时点**；保存原计划与 Recorder，另冻结修复版，将该报告和依赖它的 delta4 行设未知，除非取得精确首发版本。不能凭本卡断言全历史发布日、因子有效或 SOXX 超额。

证据：[CFTC 特殊公告](https://www.cftc.gov/MarketReports/CommitmentsofTraders/HistoricalSpecialAnnouncements/index.htm)、官方 [12-23 TFF 周报](https://www.cftc.gov/sites/default/files/files/dea/cotarchives/2025/futures/financial_lf122325.htm)。
